# Fine-tune Qwen3.5 in Unsloth Studio: support messages to ticket JSON

In this workshop you teach a small model a real business task, **turning a messy customer support message into a ticket with a fixed JSON schema**, without writing training code. You train in **Unsloth Studio**, Unsloth's web UI, on an AMD Radeon GPU, then come back to this notebook to measure what the fine-tuning bought you.

```
"Case #99570. Customer Elena reports the rice cooker is not the        {"order_id": "ORD-10968", "customer": "Elena",
 model I ordered (ref ORD-10968). Order placed 24 January 2026.   ->    "phone": "(815) 781-9664", "product": "rice cooker",
 Total was USD 867.25. Call me on (815) 781-9664."                       "category": "wrong_item", "purchase_date": "2026-01-24",
                                                                         "amount": 867.25}
```

| Part | Where | Time |
|---|---|---|
| 1. Look at the data | this notebook | 5 min |
| 2. Train and compare | Unsloth Studio | 20 min |
| 3. Score it | this notebook | 10 min |
| 4. Export | Unsloth Studio | 5 min |

**Why supervised fine-tuning (SFT)?** Studio trains with SFT: you show the model input and the exact answer you want, and it learns to imitate. That fits this task, where every example has one right answer. (The Sudoku notebook in this repo uses GRPO, reinforcement learning with reward functions, which Studio's web UI does not offer.)

## 1. Look at the data

`data/support_tickets_train.jsonl` holds 1,000 synthetic support messages, each paired with the ticket we want. `data/support_tickets_eval.jsonl` holds 100 more the model never trains on, for scoring in part 3. Both come from `scripts/generate_support_ticket_dataset.py`; the next cell regenerates them if they are missing.

In [ ]:
import collections, json, pathlib, subprocess, sys

DATA = pathlib.Path("data")
if not (DATA / "support_tickets_train.jsonl").exists():
    subprocess.run([sys.executable, "scripts/generate_support_ticket_dataset.py"], check=True)

train = [json.loads(line) for line in open(DATA / "support_tickets_train.jsonl")]
evalset = [json.loads(line) for line in open(DATA / "support_tickets_eval.jsonl")]
print(f"{len(train)} training rows, {len(evalset)} held-out rows\n")

for row in train[:3]:
    user, ticket = row["messages"][0]["content"], row["messages"][1]["content"]
    print(user, "\n  ->", ticket, "\n")

print(collections.Counter(json.loads(r["messages"][1]["content"])["category"] for r in train))

A few things to notice, because they are what the model has to learn:

- **The instruction is one short line.** It never lists the keys, the seven categories, the date format, or that missing values are `null`. The model has to pick all of that up from the examples, which is exactly what SFT is for. A shorter prompt is also cheaper to run at scale.
- **Some of it is real work, not copying.** "isn't what I expected" means `return_request`; `Jun 5, 2026` and `24 January 2026` both become `2026-06-05`-style dates; `USD 1,007.56` becomes the number `1007.56`.
- **Distractors.** Case numbers and tracking numbers look like IDs but never belong in the ticket.
- **Chat format.** Each row is a `messages` list of user and assistant turns. Studio detects it and trains through the model's own chat template, the same one Studio's Chat page uses, so what you test in Chat is what you trained.

## 2. Train it in Unsloth Studio

Open Studio in a new browser tab. Your facilitator gives you the address; on your own machine, install it with `curl -fsSL https://unsloth.ai/install.sh | sh` and start it with `unsloth studio`.

![Studio home](docs/studio-screenshots/01-studio-home.png)

### 2a. Pick the model

Go to **Train**. Under **Model**, open the picker, switch to the **Hugging Face** tab and choose `unsloth/Qwen3.5-2B`, Alibaba's 2-billion-parameter Qwen3.5. It is small enough to train in minutes and good at following formats.

![Model picker](docs/studio-screenshots/02-model-picker.png)

Set **Method** to **LoRA**. LoRA freezes the base model and trains small adapter matrices next to it, here about 1% of the parameters. QLoRA does the same on a 4-bit copy of the model to save memory; on a Radeon GPU with plenty of memory, 16-bit LoRA is the simpler, more dependable choice.

![Training method](docs/studio-screenshots/03-training-method.png)

### 2b. Upload the data

Download `data/support_tickets_train.jsonl` from this notebook's file browser (right-click, **Download**), then drop it on **Upload a local file**. Click **View dataset** to check Studio read it as a conversation dataset: it should say *Detected ChatML conversation column*.

![Dataset preview](docs/studio-screenshots/04-dataset-preview.png)

### 2c. Set the run and start it

Under **Parameters**, name the project `support-tickets` and set **Max Steps** to **60**. Leave everything else at Studio's defaults. Check the **Run preview** on the right, then click **Start Training**.

![Run configured](docs/studio-screenshots/05-run-configured.png)

### 2d. Watch it learn

Studio moves to **Current Run**. The first minute or two is loading the model and formatting the data; the very first run on a machine also compiles a small ROCm kernel for Qwen3.5's attention layers, which takes a few minutes once. After that, the step counter moves and the charts fill in.

![Training in progress](docs/studio-screenshots/06-training-in-progress.png)

What to look for in **Training Loss**: a steep drop over the first 10 to 15 steps, then a long flat tail near zero. The drop is the model picking up the schema; the tail is it polishing the details. A loss that does not fall at all usually means the data was not read as conversations (go back to 2b).

![Training complete](docs/studio-screenshots/07-training-complete.png)

### 2e. Compare base and fine-tuned in Chat

When it finishes, click **Compare in Chat**. Studio loads the base model and your fine-tuned one side by side and sends each message to both.

**Turn Thinking off** (the lightbulb next to the send button) before you send anything. With Thinking on, Qwen3.5 reasons first and the prompt no longer looks like the training examples, so the fine-tuned model drifts.

Paste in a message it has never seen; the next cell prints one.

In [ ]:
# A held-out message to paste into Compare in Chat, and the ticket we hope to get back.
sample = evalset[11]
print(sample["messages"][0]["content"])
print("\nexpected:", sample["messages"][1]["content"])

![Compare in Chat](docs/studio-screenshots/08-compare-in-chat.png)

The base model writes a plausible ticket, but with its own keys, its own nesting and its own idea of what matters. The fine-tuned model answers with exactly your schema. One example is an anecdote, though, so next we measure.

## 3. Score it

This is the part Studio cannot do for you: decide whether the fine-tuned model is actually better, on data it has not seen. We compare three setups on the held-out tickets:

1. **Base model, short prompt**: the same one-line instruction the training data used.
2. **Base model, schema prompt**: what you would do without fine-tuning, spelling out every key, the category list, the date format and the null rule in the prompt.
3. **Fine-tuned model, short prompt**.

Each answer gets three scores: whether it is a **bare JSON** object (no code fence or chatter, so code can consume it directly), the share of the seven **fields correct**, and the share of **whole tickets exactly right**.

First, find the adapter Studio saved. Its path is at the top of the **Current Run** page ("Model saved to ..."); the cell below picks the newest `support-tickets` run automatically. If Studio runs on a different machine from this notebook, use **Export > LoRA Only** in part 4, copy the folder here, and set `ADAPTER_DIR` by hand.

In [ ]:
import glob, os

roots = [os.environ.get("UNSLOTH_STUDIO_HOME", ""), os.path.expanduser("~/.unsloth/studio")]
runs = sorted({p for r in roots if r for p in glob.glob(os.path.join(r, "outputs", "*project-support-tickets*"))},
              key=os.path.getmtime)
ADAPTER_DIR = runs[-1] if runs else None   # or the path from "Model saved to ..."
print(ADAPTER_DIR)

In [ ]:
# Load the base model with your adapter on top. The same model answers as "base"
# with the adapter switched off, so only one copy is in memory.
sys.path.insert(0, "scripts")
from score_tickets import SCHEMA_PROMPT, generate, score
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(ADAPTER_DIR, load_in_4bit = False)
FastModel.for_inference(model);

In [ ]:
# 30 held-out tickets per setup; raise N for a tighter estimate if you have time.
N = 30
rows = evalset[:N]
short = [r["messages"][0]["content"] for r in rows]
schema = [SCHEMA_PROMPT + "\n\n" + p.split("\n\n", 1)[1] for p in short]

answers = {}
with model.disable_adapter():
    answers["base, short prompt"] = generate(model, tokenizer, short)
    answers["base, schema prompt"] = generate(model, tokenizer, schema)
answers["fine-tuned, short prompt"] = generate(model, tokenizer, short)

import pandas as pd
pd.DataFrame({name: score(rows, a) for name, a in answers.items()}).T.style.format("{:.0%}")

In [ ]:
# The same held-out message through all three.
i = 3
print(short[i].split("\n\n", 1)[1], "\n")
for name, a in answers.items():
    print(f"--- {name}\n{a[i]}\n")

How to read it:

- The **base model with the short prompt** has no idea what "a ticket JSON" means here. It invents its own format, so it scores zero.
- The **schema prompt** is the honest baseline: spelling everything out gets the base model most of the way, and prompt engineering is often enough. Look at what it still gets wrong; it is usually the fine print (a category that needs judgment, `null` versus a guess).
- The **fine-tuned model** learned the schema, categories and rules from examples, and applies them from a one-line prompt. In our test run it got every held-out ticket exactly right. It also costs less per call: no instructions block to send every time.

If your fine-tuned model scores lower, look at the answers it got wrong. Extra keys or rambling usually mean too few steps; a wrong category usually means phrasing the training data barely covers.

## 4. Export

Back in Studio, **Export Model** (on the Current Run page, or under **More > Export**) packages the result:

- **LoRA Only**: just the adapter, about 100 MB. Needs the base model alongside it.
- **Merged Model**: base and adapter folded into one 16-bit model, ready for any Transformers or vLLM server.
- **GGUF / Llama.cpp**: quantized single files for local runners such as llama.cpp, LM Studio and Ollama. From a GGUF, Studio can also convert to **Q4NX** for the NPU in AMD Ryzen AI processors.

![Export](docs/studio-screenshots/09-export.png)

In [ ]:
# Free the GPU memory this notebook holds before training anything else in Studio.
del model
import gc, torch
gc.collect(); torch.cuda.empty_cache()

## Where to go next

- **Your own data.** Any JSONL with a `messages` column (or Alpaca's `instruction` / `input` / `output`) drops straight into Studio. Start with a few hundred clean examples of the exact output you want.
- **A bigger model.** `unsloth/gemma-4-E4B-it` is a drop-in swap with more headroom for harder tasks.
- **Rewards instead of examples.** When there is no single right answer but you can *score* one, see the GRPO Sudoku notebook in this repo.